# Chapter 6 Walkthrough: HCP Targeting

This executed notebook builds the Chapter 6 artifacts at the December 31, 2024 cutoff. Run `ch06_hcp/scripts/generate_ch06_data.py` before rebuilding the notebook.


In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / "ch06_hcp").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Run this notebook inside the repository.")
    ROOT = ROOT.parent

SCRIPT_DIR = ROOT / "ch06_hcp" / "scripts"
sys.path.insert(0, str(SCRIPT_DIR))

analysis_module = importlib.import_module("run_analysis")
targeting = importlib.import_module("targeting")
referral = importlib.import_module("referral_network")
segmentation = importlib.import_module("segmentation")

results = analysis_module.run_analysis(ROOT)

headline = pd.Series({
    "Journey patients": results["attribution_comparison"].patient_id.nunique(),
    "Eligible-roster patients": results["patient_hcp"].patient_id.nunique(),
    "Eligible HCPs": results["hcp_features"].npi.nunique(),
})
display(headline.to_frame("count"))


,count
Journey patients,6393
Eligible-roster patients,1556
Eligible HCPs,158


The eligible roster covers 1,556 patients and 158 HCPs. The remaining journey patients stay outside this field-planning artifact.


## 1. Attribution sensitivity


In [2]:
agreement = results["attribution_summary"].copy()
agreement["agreement_rate"] = agreement["agreement_rate"].map(
    lambda value: f"{value:.1%}"
)
display(agreement)
display(
    results["attribution_comparison"].query(
        "patient_id == 'PAT02034'"
    )
)


,comparison,patients_with_both,same_hcp,agreement_rate
0,Index vs plurality,6393,4399,68.8%
1,Index vs latest,6393,4088,63.9%
2,All 3 rules,6393,4005,62.6%


,patient_id,index_npi,plurality_npi,latest_npi,all_rules_agree
635,PAT02034,9000000280,9000000280,9000000280,True


All 3 attribution rules agree for 63.4% of patients. PAT02034 remains assigned to HCP0280 under every rule.


## 2. HCP evidence and concentration


In [3]:
columns = [
    "npi", "account_id", "cohort_patients", "treated_patients",
    "roventra_starts", "competitor_treated", "untreated_mature",
    "review_opportunity", "contact_permission_status",
]
display(
    results["hcp_features"].sort_values(
        ["cohort_patients", "npi"], ascending=[False, True]
    )[columns].head(10)
)
display(results["decile_summary"].head())


,npi,account_id,cohort_patients,treated_patients,roventra_starts,competitor_treated,untreated_mature,review_opportunity,contact_permission_status
93,9000000430,ACC189,36,9,2,7,25,32,Allowed
105,9000000469,ACC121,34,10,9,1,21,22,Opt-out
35,9000000162,ACC062,33,13,8,5,19,24,Opt-out
99,9000000447,ACC216,32,12,5,7,17,24,Opt-out
5,9000000026,ACC226,28,11,8,3,15,18,Allowed
121,9000000537,ACC079,28,6,2,4,18,22,Opt-out
49,9000000217,ACC224,27,8,4,4,16,20,Allowed
117,9000000516,ACC167,27,9,6,3,18,21,Allowed
102,9000000460,ACC219,26,10,7,3,15,18,Allowed
86,9000000389,ACC155,24,8,4,4,15,19,Allowed


,opportunity_decile,hcps,cohort_patients,review_opportunity,cumulative_hcp_share,cumulative_opportunity_share
0,1,12,279,216,0.107143,0.265683
1,2,11,157,123,0.205357,0.416974
2,3,11,127,100,0.303571,0.539975
3,4,11,109,84,0.401786,0.643296
4,5,11,83,68,0.500000,0.726937


The highest-volume rows include opt-outs. The first 30% of HCPs capture 55.2% of review opportunity.


![Top 20 HCPs ranked by review opportunity. Blue bars show review opportunity for Allowed HCPs, red bars for Opt-out HCPs, gray for Unknown. Light blue shows remaining attributed patients.](assets/figures/figure_6_1_volume_diagnostic.svg)

*Figure 6.1. Review opportunity ranked highest to lowest, colored by contact permission. An HCP near the top with a red bar holds substantial opportunity but cannot be worked through the field channel in this cycle. Synthetic data.*

![Line chart starting at the origin showing cumulative review opportunity captured as contactable HCP share increases, ranked by review opportunity, with a dashed diagonal reference line.](assets/figures/figure_6_2_cumulative_capture.svg)

*Figure 6.2. The curve starts at (0%, 0%) and rises steeply. The top 30% of contactable HCPs by review opportunity account for 54% of total contactable opportunity. The dashed diagonal shows what equal distribution would look like. Synthetic data.*


## 3. Referral pathways


In [4]:
display(results["referral_edges"].head(10))
stable = results["referral_metrics"].merge(
    results["referral_stability"][[
        "npi", "bootstrap_top20_frequency", "window_rank_range",
    ]],
    on="npi",
)
display(stable.head(15))


,source_npi,destination_npi,source_specialty,destination_specialty,source_account_id,destination_account_id,region,unique_patients,referral_episodes,median_transition_days,first_referral_date,last_referral_date,path_cost
0,9000000578,9000000258,Primary Care,Endocrinology,ACC142,ACC164,Midwest,22,22,25.0,2024-01-09,2024-10-29,0.045455
1,9000000417,9000000164,Primary Care,Endocrinology,ACC126,ACC109,West,20,20,40.0,2024-01-22,2024-11-28,0.050000
2,9000000460,9000000567,Primary Care,Endocrinology,ACC219,ACC030,South,20,20,24.5,2024-01-29,2024-12-18,0.050000
3,9000000033,9000000302,Primary Care,Endocrinology,ACC044,ACC090,South,19,19,32.0,2024-02-02,2024-12-24,0.052632
4,9000000265,9000000409,Primary Care,Endocrinology,ACC148,ACC164,Midwest,19,19,27.0,2024-02-13,2024-12-23,0.052632
5,9000000520,9000000127,Primary Care,Endocrinology,ACC110,ACC073,South,19,19,29.0,2024-01-16,2024-12-26,0.052632
6,9000000020,9000000409,Primary Care,Endocrinology,ACC068,ACC164,Midwest,18,18,37.0,2024-02-25,2024-11-28,0.055556
7,9000000128,9000000567,Primary Care,Endocrinology,ACC160,ACC030,South,18,18,31.5,2024-02-24,2024-12-26,0.055556
8,9000000470,9000000217,Primary Care,Endocrinology,ACC068,ACC224,Midwest,18,18,29.0,2024-01-21,2024-10-26,0.055556
9,9000000565,9000000217,Primary Care,Endocrinology,ACC099,ACC224,Midwest,18,18,32.5,2024-02-19,2024-12-26,0.055556


,npi,specialty,account_id,region,unique_sources,unique_destinations,patients_received,patients_referred,betweenness_centrality,pathway_patient_volume,pathway_breadth,bootstrap_top20_frequency,window_rank_range
0,9000000217,Endocrinology,ACC224,Midwest,6,2,72,15,0.000626,87,8,1.0000,1
1,9000000567,Endocrinology,ACC030,South,5,2,66,14,0.000521,80,7,1.0000,1
2,9000000127,Endocrinology,ACC073,South,7,2,57,13,0.000730,70,9,1.0000,2
3,9000000170,Endocrinology,ACC132,Northeast,8,2,56,13,0.000834,69,10,0.9875,3
4,9000000204,Endocrinology,ACC153,South,6,2,55,9,0.000521,64,8,1.0000,2
5,9000000215,Endocrinology,ACC183,South,7,1,56,8,0.000313,64,8,0.9875,3
6,9000000207,Endocrinology,ACC094,Northeast,4,2,46,16,0.000417,62,6,1.0000,1
7,9000000258,Endocrinology,ACC164,Midwest,4,2,49,12,0.000417,61,6,1.0000,4
8,9000000550,Endocrinology,ACC179,West,5,2,50,9,0.000469,59,7,1.0000,3
9,9000000636,Endocrinology,ACC059,Northeast,7,1,47,11,0.000313,58,8,0.9875,3


The referral output is a pathway-context artifact. Stability comes from transition-window comparison and patient-level bootstrap resampling.


![Schematic referral graph illustrating directed edges, patient counts, and betweenness centrality.](assets/figures/figure_6_3_referral_schematic.svg)

*Figure 6.3. Conceptual illustration of the referral graph structure used in this chapter. Nodes A-C are Primary Care physicians (blue), node D is the Endocrinologist hub (gold), and nodes E-F are Cardiologists (green). Arrow width reflects patient count on each edge. Node D has the highest betweenness centrality because it bridges multiple upstream sources to downstream specialists.*

![Directed account-centered referral network with patient counts on each edge.](assets/figures/figure_6_4_referral_network.svg)

*Figure 6.4. The ego network shows the highest-betweenness HCP and the ten strongest referral edges connected to that physician. Patient count labels each edge. Synthetic data.*


## 4. Scientific role evidence


In [5]:
candidates = results["kol_profiles"].loc[
    results["kol_profiles"]["kol_candidate"]
]
display(candidates[[
    "npi", "specialty_1", "research_percentile",
    "leadership_percentile", "practice_expertise_percentile",
    "peer_connection_percentile", "proposed_role",
    "role_fit_score", "evidence_confidence",
]].head(15))
display(results["kol_validation"])
display(results["kol_transparency_review"].head())


,npi,specialty_1,research_percentile,leadership_percentile,practice_expertise_percentile,peer_connection_percentile,proposed_role,role_fit_score,evidence_confidence
0,9000000105,Cardiology,100.000000,80.000000,25.000000,55.000000,National scientific leader,89.0,High
1,9000000206,Endocrinology,100.000000,8.333333,34.782609,15.384615,Evidence-generation collaborator,77.2,High
2,9000000211,Endocrinology,100.000000,36.363636,80.000000,70.833333,Evidence-generation collaborator,93.0,High
3,9000000237,Primary Care,100.000000,75.000000,77.777778,91.666667,Evidence-generation collaborator,92.2,High
4,9000000363,Endocrinology,100.000000,100.000000,100.000000,66.666667,Evidence-generation collaborator,100.0,High
5,9000000441,Primary Care,100.000000,84.615385,77.777778,26.190476,Evidence-generation collaborator,92.2,High
6,9000000512,Cardiology,100.000000,31.250000,94.444444,92.500000,Evidence-generation collaborator,98.1,High
7,9000000562,Cardiology,100.000000,40.000000,89.473684,75.000000,Evidence-generation collaborator,96.3,High
8,9000000633,Primary Care,100.000000,92.592593,4.545455,59.375000,National scientific leader,95.9,High
9,9000000366,Endocrinology,96.153846,4.166667,39.130435,15.384615,Evidence-generation collaborator,76.2,High


,validation_measure,value
0,KOL candidates,83.000000
1,Reviewed candidates,79.000000
2,Proposed role match rate,0.582278
3,Reviewer confirmation rate,0.639241
4,Reviewer decision kappa,0.426150


,npi,proposed_role,review_status_x,total_payment_amount,payment_records,payment_categories,latest_payment_year,review_status_y,transparency_use
0,9000000105,National scientific leader,Medical-affairs review required,85575.71,7,Education/Training | Research Grants | Speakin...,2024.0,Transparency review only,Disclosure context only; excluded from scienti...
1,9000000206,Evidence-generation collaborator,Medical-affairs review required,0.00,0,NaN,NaN,NaN,Disclosure context only; excluded from scienti...
2,9000000211,Evidence-generation collaborator,Medical-affairs review required,0.00,0,NaN,NaN,NaN,Disclosure context only; excluded from scienti...
3,9000000237,Evidence-generation collaborator,Medical-affairs review required,0.00,0,NaN,NaN,NaN,Disclosure context only; excluded from scienti...
4,9000000363,Evidence-generation collaborator,Medical-affairs review required,0.00,0,NaN,NaN,NaN,Disclosure context only; excluded from scienti...


Scientific role fit and payment transparency remain separate. Medical affairs owns the role review.


![Four scatter plots, one per proposed scientific role, showing candidate positions on that role's two primary evidence dimensions. Gray dots show candidates from other roles for context.](assets/figures/figure_6_5_kol_evidence_matrix.svg)

*Figure 6.5. Each panel focuses on one proposed role and plots the two dimensions that dominate its fit formula. Gray dots are candidates assigned to other roles. The same candidate can look strong or weak depending on which role lens is applied. Synthetic data.*


## 5. K-means engagement profiles


In [6]:
display(results["cluster_evaluation"])
display(results["segment_profiles"])
display(results["segment_policy_comparison"])


,k,inertia,silhouette,minimum_cluster_size,minimum_cluster_share,seed_stability_ari,bootstrap_stability_ari,selection_score,operational_size_pass,selected
0,3,13.929700,0.641224,14,0.250000,1.000000,0.947400,1.228074,True,False
1,4,3.515723,0.763294,9,0.160714,1.000000,1.000000,1.363294,True,True
2,5,2.902478,0.712550,4,0.071429,1.000000,0.918132,1.239702,False,False
3,6,2.490435,0.508611,4,0.071429,0.925109,0.810973,0.990251,False,False


,cluster_id,hcp_count,cohort_patients,opportunity_rate,roventra_share,access_signal_rate,recent_contacts,productive_contact_rate,evidence_need_score,access_resource_score,digital_response_rate,field_response_rate,segment_name,engagement_pattern
0,0,9,14.111111,0.670029,0.596152,0.005556,1.000000,0.394180,0.800000,0.589333,0.770111,0.236667,C0: Digital evidence seeker,"Approved digital evidence, then field review"
1,1,14,14.571429,0.755483,0.495040,0.001984,1.857143,0.461905,0.309214,0.347214,0.210071,0.813929,C1: Field maintenance,Maintenance field follow-up
2,2,22,13.181818,0.777571,0.456602,0.004132,1.227273,0.357035,0.793909,0.619273,0.225773,0.820773,C2: Field evidence builder,Field evidence discussion
3,3,11,11.000000,0.702138,0.556061,0.000000,0.909091,0.513420,0.292818,0.330455,0.766909,0.233909,C3: Digital maintenance,"Digital maintenance, then field review"


,segment_name,Access-resource need,Balanced follow-up,Digital evidence seeker,Established adopter,Field evidence builder
0,C0: Digital evidence seeker,1,0,8,0,0
1,C1: Field maintenance,0,13,0,1,0
2,C2: Field evidence builder,5,0,0,0,17
3,C3: Digital maintenance,0,8,0,3,0


The selected 4-cluster solution has silhouette 0.763, seed ARI 1.000, bootstrap ARI 1.000, and minimum cluster size 9.


![Line chart comparing silhouette, seed ARI, and bootstrap ARI for candidate cluster counts.](assets/figures/figure_6_6_cluster_validation.svg)

*Figure 6.6. k=4 achieves the best silhouette score, seed ARI, and bootstrap ARI among solutions that pass the minimum cluster-size gate. Synthetic data.*

![2x2 small-multiples bar charts showing each engagement profile's evidence-need, access-need, digital-response, and field-response scores.](assets/figures/figure_6_7_segment_profiles.svg)

*Figure 6.7. Each panel is one engagement profile. The dashed line marks 0.5 (mid-range). C0 and C2 both show high evidence-need bars but diverge on which response channel is tall; C1 and C3 both show lower evidence-need bars but split the same way on channel. Synthetic data.*


## 6. HCP call plan


In [7]:
display(results["call_plan"])
display(results["plan_comparison"])


,cycle_start,cycle_end,territory,account_id,parent_account_id,account_name,npi,specialty,account_action,hcp_action,engagement_pattern,segment_name,recommended_calls,hcp_review_opportunity,recent_contacts,permission_status,reason_code,reason,territory_cycle_capacity
0,2025-01-01,2025-01-28,T01,ACC224,SYS-MID-09,Michigan Care 224,9000000217,Endocrinology,Increase priority,Prioritize,Maintenance field follow-up,C1: Field maintenance,2,20,1,Allowed,PRIORITIZE_REVIEW_OPPORTUNITY,Permitted review opportunity and adoption belo...,48
1,2025-01-01,2025-01-28,T01,ACC056,SYS-NOR-09,Pennsylvania Care 056,9000000136,Primary Care,Increase priority,Prioritize,Field evidence discussion,C2: Field evidence builder,2,11,1,Allowed,PRIORITIZE_REVIEW_OPPORTUNITY,Permitted review opportunity and adoption belo...,48
2,2025-01-01,2025-01-28,T03,ACC034,SYS-SOU-11,Florida Care 034,9000000273,Primary Care,Increase priority,Prioritize,Field review,Not clustered,2,6,0,Allowed,PRIORITIZE_REVIEW_OPPORTUNITY,Permitted review opportunity and adoption belo...,56
3,2025-01-01,2025-01-28,T04,ACC155,SYS-WES-12,Arizona Care 155,9000000389,Cardiology,Increase priority,Prioritize,"Digital maintenance, then field review",C3: Digital maintenance,2,19,1,Allowed,PRIORITIZE_REVIEW_OPPORTUNITY,Permitted review opportunity and adoption belo...,48
4,2025-01-01,2025-01-28,T04,ACC219,SYS-SOU-04,Florida Care 219,9000000460,Primary Care,Maintain,Maintain,Field evidence discussion,C2: Field evidence builder,1,18,0,Allowed,MAINTAIN_ESTABLISHED,Permitted evidence with adoption at or above t...,48
5,2025-01-01,2025-01-28,T05,ACC124,SYS-NOR-05,New York Care 124,9000000035,Primary Care,Increase priority,Prioritize,Field evidence discussion,C2: Field evidence builder,2,6,1,Allowed,PRIORITIZE_REVIEW_OPPORTUNITY,Permitted review opportunity and adoption belo...,52
6,2025-01-01,2025-01-28,T06,ACC189,SYS-MID-10,Michigan Care 189,9000000430,Cardiology,Increase priority,Prioritize,Maintenance field follow-up,C1: Field maintenance,2,32,0,Allowed,PRIORITIZE_REVIEW_OPPORTUNITY,Permitted review opportunity and adoption belo...,56
7,2025-01-01,2025-01-28,T06,ACC109,SYS-WES-02,Arizona Care 109,9000000164,Endocrinology,Increase priority,Prioritize,"Approved digital evidence, then field review",C0: Digital evidence seeker,2,13,0,Allowed,PRIORITIZE_REVIEW_OPPORTUNITY,Permitted review opportunity and adoption belo...,56
8,2025-01-01,2025-01-28,T06,ACC005,SYS-SOU-06,Florida Care 005,9000000498,Cardiology,Increase priority,Prioritize,Field evidence discussion,C2: Field evidence builder,2,7,1,Allowed,PRIORITIZE_REVIEW_OPPORTUNITY,Permitted review opportunity and adoption belo...,56
9,2025-01-01,2025-01-28,T06,ACC005,SYS-SOU-06,Florida Care 005,9000000051,Cardiology,Increase priority,Prioritize,Field evidence discussion,C2: Field evidence builder,1,6,0,Allowed,PRIORITIZE_REVIEW_OPPORTUNITY,Permitted review opportunity and adoption belo...,56


,plan,selected_hcps,contact_permitted,held_or_unknown,review_opportunity,recent_contacts
0,Top 30 by patient volume,30,30,0,397,43
1,Gated 4-week field plan,11,11,0,143,6


The HCP call plan contains 11 permitted HCPs and 20 recommended calls. Each row keeps site account context for routing.


## 8. Export the evidence package


In [8]:
output_dir = ROOT / "ch06_hcp" / "assets" / "generated_outputs"
analysis_module.write_outputs(results, output_dir, ROOT)
print(f"Wrote {len(results)} CSV artifacts and manifest.json")


Wrote 32 CSV artifacts and manifest.json


The package carries analysis date, source hashes, rule version, decision boundaries, and output contracts.
